In [ ]:
import torch
import numpy as np
import custom_gym
import matplotlib
# matplotlib.use('Agg')
%matplotlib inline

from dynamic_systems_rl import PointMass

from stable_baselines3.common.env_checker import check_env
from stable_baselines3 import PPO, SAC, DDPG

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
order=2

In [ ]:

dt=0.01;
x_obst = []#[torch.tensor([0.,-0.4])]#[torch.tensor([0.,-0.5]).to(device)] #[torch.tensor([0.35,0.5]).to(device),torch.tensor([0.35,-0.3]).to(device),torch.tensor([-0.1,-0.55]).to(device)]
r_obst = []#[0.2]#[0.2] #[0.2]*len(x_obst)
agent = PointMass(order=order, dt=dt, dim=2,
                x_obst=x_obst,r_obst=r_obst,
                w_obst=0., w_goal=1, w_action= 1)

In [ ]:
if order==1:
    reset_bound = np.array([1,1]) 
else:
    reset_bound = np.array([1,1,0.1,0.1]) 
env = custom_gym.CreateGymEnv(agent, 
                              reset_bound=reset_bound,     
                              dt=dt)

In [ ]:
# It will check your custom environment and output additional warnings if needed
check_env(env)

In [ ]:
# Instantiate the agent and Train the agent
policy_kwargs = dict(net_arch=[128, 128])

model = PPO(
    "MlpPolicy",
    env,
    gamma=0.99,
    use_sde=True,
    sde_sample_freq=4,
    learning_rate=1e-3,
    verbose=1,
    policy_kwargs=policy_kwargs
)


In [ ]:
def policy(state):
    action, _ = model.predict(state, deterministic=True)
    return action

def callback(agent,policy, dt, order=order, dim=2, device='cpu',plt=False):
    print("Testing....")
    state = 2*(-0.5+torch.rand((100,dim*order)))
    history = []
    traj = state[:,:2].clone()[:,None,:]
    cum_reward = torch.tensor([0.]*state.shape[0]).to(device)
    for i in range(5000):
        action =  torch.from_numpy(policy(state)) #lqr_policy(state)#
        r = agent.reward_state_action(state,action)
        cum_reward+=r
        state = agent.forward_simulate(state,action,dt)
        position = state[:,:2]
        traj = torch.concat((traj,position[:,None,:2]),dim=1)
    
    dist_straight = torch.linalg.norm(traj[:,0,:]-traj[:,-1,:],dim=-1).view(-1,1)
    d_traj = torch.linalg.norm(traj[:,1:,:]-traj[:,:-1,:],dim=-1).sum(dim=-1).view(-1,1)
    mu_i = (dist_straight/(1e-6+d_traj))
    
    dist_final = torch.linalg.norm(traj[:,-1,:],dim=-1).view(-1)
    is_success = (dist_final<0.05)
    success_rate = torch.sum(is_success)/len(is_success)
    mu = ((mu_i.view(-1)*is_success).sum()/(1e-9+torch.sum(is_success)))**2
    print("distance metric: ", mu)
    print("sucess_rate: ", success_rate)
    if plt:
        from matplotlib import pyplot as plt
        from plot_utils import plot_point_mass
        plt=plot_point_mass(traj.to('cpu'),x_target=torch.tensor([0,0]).to('cpu'), x_obst=[x.to('cpu') for x in x_obst], r_obst=r_obst, figsize=5)
        plt.show()
    out = (success_rate, mu, cum_reward.mean().to("cpu"))
    return out


In [ ]:
import time
T = 0
for i in range(10):
    t1 = time.time()
    model.learn(total_timesteps=int(1e5))
    t2 = time.time()
    T = T+ (t2-t1)
    print("Time Taken Per Iteration: ", t2-t1)
    print("Total time: ", T)
    model.save('pm_acc_sac'+str(i))
    
    N=5
    S = torch.empty(N); Mu=  torch.empty(N); CumR =   torch.empty(N);
    for j in range(5):
        plt_ = False if j<(N-1) else True
        s,mu, r_cum= callback(agent=agent,policy=policy, 
                              order=order, dim=2, dt=dt, plt=plt_)
        S[j] = s; Mu[j] = mu; CumR[j] = r_cum;
    print("mu, mean:{}, std:{}".format(torch.mean(Mu),torch.std(Mu)))
    print("s, mean:{}, std:{}".format(torch.mean(S),torch.std(S)))
    print("cum_r, mean:{}, std:{}".format(torch.mean(CumR),torch.std(CumR)))
        